# Week 2 Day 1: is the difference real?

*Feyisa Diba's version of the Week 2 Day 1 assignment. The write-up, with my results and what I concluded, is in [WEEK2_DAY1_LANGFUSE.md](WEEK2_DAY1_LANGFUSE.md).*

The assignment asks three questions about the concise policy wording from Week 1 Day 4, `data/policy_wording_short.md`, used with the payout tool:

1. Does it keep the accuracy of the full wording?
2. Is it cheaper or faster?
3. If a wrongful decline costs UnderCover more or less than `WRONGFUL_DECLINE`, does that change which wording we would choose?

Other students have answered them, and their answers disagree, often because one run of 25 claims is noisy. This notebook tries to separate the difference from the noise:

- **Every claim counts.** Langfuse's `run_experiment` leaves out a claim whose task fails, and scores the run on the rest. In my first run of the lab, two claims hit OpenRouter's rate limit, and the baseline was scored on 23 claims. Here the task waits for the rate limit, retries, and counts a claim that still fails as a referral, as the lab does for a claim with no decision.
- **Three runs of each experiment**, with the time, tokens, cached tokens and cost of every claim.
- **A paired bootstrap** gives a 95% interval for each difference between the wordings, to show whether a gap is bigger than the noise.
- **`WRONGFUL_DECLINE` stays at the agreed $1,500.** We find the break-even figure where the choice of wording flips, score five other figures and a scenario where the cost grows with the claim, and add the new costs to each run in Langfuse.

It calls GPT-6 Luna through OpenRouter: about a cent a run, so about 10 cents for the nine runs. They take 30 to 40 minutes, because new OpenRouter accounts are limited to 20 requests a minute. Each finished run is saved to `week2day1_langfuse_results.csv` and isn't run again, so the analysis can be run again for free.

In [ ]:
import json
import os
import sys
import time
from collections import deque
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler

# This notebook is in community_contributions/feyisa_diba. The payout rules, the data and .env are found from the repo's root
REPO_ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "labs" / "week2day1_payout.py").is_file())
sys.path.insert(0, str(REPO_ROOT / "labs"))
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter and Langfuse

The keys come from the `.env` file in the `undercover` folder, as in the lab.

In [ ]:
load_dotenv(REPO_ROOT / ".env", override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
print("OpenRouter API key found" if openrouter_api_key else "No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

langfuse = get_client()
print("Connected to Langfuse" if langfuse.auth_check() else "No Langfuse keys found. Add them to the .env file")

## The claims and the two wordings

The lab's 25 harder claims, with the handlers' decisions.

In [ ]:
HARDER = REPO_ROOT / "data" / "evals" / "harder"


def load(name):
    """Read one of the JSON files of the harder claims."""
    return json.loads((HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (REPO_ROOT / "data" / "policy_wording.md").read_text(encoding="utf-8")
short_wording = (REPO_ROOT / "data" / "policy_wording_short.md").read_text(encoding="utf-8")

full_words, short_words = len(policy_wording.split()), len(short_wording.split())
print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"Full wording: {full_words:,} words. Short wording: {short_words:,} words ({short_words / full_words - 1:+.0%})")

## The three agents

The instructions, the prompt, the tools and the model are the lab's. The short wording has no numbered sections, so its agent's instructions say "the Payment section" where the lab's say "section 10", as Andras Nemes did. That is the only other change.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

SHORT_INSTRUCTIONS = INSTRUCTIONS.replace("section 10", "the Payment section")
assert SHORT_INSTRUCTIONS != INSTRUCTIONS

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="none", timeout=120, max_retries=5)

AGENTS = {
    "Baseline": create_agent(model=luna, tools=[record_decision], system_prompt=f"{INSTRUCTIONS}\n{policy_wording}\n"),
    "Payout tool": create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n"),
    "Short wording + payout tool": create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{SHORT_INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{short_wording}\n"),
}

## The cost of mistakes

The lab's cost matrix. `mistake_cost` takes the cost of a wrongful decline as an argument, so the same answers can be scored again without calling the model. It can be a figure, or a function of the handler's decision for a cost that grows with the claim.

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected, wrongful_decline=None):
    """What the agent's answer costs UnderCover, against the handler's decision.

    wrongful_decline is the cost of declining a claim the handler approved or referred:
    WRONGFUL_DECLINE when it's None, otherwise a figure, or a function of the handler's decision.
    """
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        if wrongful_decline is None:
            return WRONGFUL_DECLINE
        return wrongful_decline(expected) if callable(wrongful_decline) else wrongful_decline
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()

## Every claim counts

Two changes to the lab's task. Both are about failures, not about the agent.

**Langfuse leaves out a claim that fails.** If a task raises an error, `run_experiment` logs it, drops the claim and runs the run evaluators on the rest. A rate-limited claim then vanishes from the denominator instead of counting against the agent, and Langfuse still shows the run with all 25 items. Here `decide` never raises. It retries, and a claim that still fails counts as a referral, as the lab counts a claim the agent didn't decide. The `failed` column flags it.

**OpenRouter limits new accounts to 20 requests a minute** for GPT-6 Luna. `decide` waits until the last minute's model calls leave room for another claim. The wait comes before the timer starts, so it isn't part of the time per claim.

The time per claim is the wall-clock time of the whole agent loop: every model call and tool call, including any retries inside the OpenAI client.

In [ ]:
RATE_LIMIT = 18  # model calls a minute, under OpenRouter's 20 for new accounts
CALLS_PER_CLAIM = 4  # room to leave for the next claim: most claims take 2 or 3 model calls
recent_calls = deque()  # (when a claim finished, the model calls it made)
langfuse_handler = CallbackHandler()


def wait_for_rate_limit():
    """Wait until the model calls of the last minute leave room for another claim."""
    while True:
        while recent_calls and time.time() - recent_calls[0][0] > 60:
            recent_calls.popleft()
        if sum(calls for _, calls in recent_calls) + CALLS_PER_CLAIM <= RATE_LIMIT:
            return
        time.sleep(1)


def usage(messages):
    """The model calls, tokens and cost of an agent's run on one claim."""
    replies = [message for message in messages if message.type == "ai"]
    details = [reply.usage_metadata or {} for reply in replies]
    return {
        "model_calls": len(replies),
        "input_tokens": sum(detail.get("input_tokens", 0) for detail in details),
        "cached_tokens": sum(detail.get("input_token_details", {}).get("cache_read", 0) for detail in details),
        "output_tokens": sum(detail.get("output_tokens", 0) for detail in details),
        "cost": sum(reply.response_metadata["token_usage"]["cost"] for reply in replies),
    }


NO_USAGE = {"model_calls": 0, "input_tokens": 0, "cached_tokens": 0, "output_tokens": 0, "cost": 0.0, "seconds": np.nan}


def decide(agent, claim, attempts=3):
    """Run an agent on one claim and return its decision, with the time, tokens and cost. It never raises."""
    claim_id = claim["claim_id"]
    for attempt in range(1, attempts + 1):
        wait_for_rate_limit()
        start = time.perf_counter()
        try:
            result = agent.invoke(
                {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
                config={"callbacks": [langfuse_handler]},
            )
        except Exception as error:
            recent_calls.append((time.time(), CALLS_PER_CLAIM))
            print(f"{claim_id}: attempt {attempt} failed with {type(error).__name__}")
            if claim_id in recorded:  # the decision was recorded before the error, and decisions are final
                return {**recorded[claim_id], **NO_USAGE, "attempts": attempt, "failed": True}
            time.sleep(30 * attempt)
            continue
        seconds = time.perf_counter() - start
        spent = usage(result["messages"])
        recent_calls.append((time.time(), spent["model_calls"]))
        answer = recorded.get(claim_id, {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
        print(f"{claim_id}: {answer['decision']} ${answer['payout']:,.2f} in {seconds:.1f}s")
        return {**answer, **spent, "seconds": seconds, "attempts": attempt, "failed": False}
    print(f"{claim_id}: failed {attempts} times, so it counts as a referral")
    return {"decision": "refer", "payout": 0.0, "rationale": "Failed", **NO_USAGE, "attempts": attempts, "failed": True}

The lab's evaluators, with more in each run's scores: the wrongful declines, the time a claim, the input tokens a claim, the share of them read from the provider's cache, and the claims that failed.

In [ ]:
USAGE_COLUMNS = ["seconds", "model_calls", "input_tokens", "cached_tokens", "output_tokens", "attempts", "failed"]


def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, and the cost of its mistake."""
    return [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores and the usage."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "llm_cost": answer["cost"],
            **{column: answer[column] for column in USAGE_COLUMNS},
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def wrongful_declines(table):
    """True for each claim the agent declined that the handler approved or referred."""
    return (table["agent"] == "decline") & (table["handler"] != "decline")


def run_scores(table):
    """The scores for a whole run: the lab's, with the wrongful declines, the time and the tokens."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "wrongful_declines": wrongful_declines(table).sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "llm_cost": table["llm_cost"].sum(),
        "seconds_a_claim": table["seconds"].mean(),
        "median_seconds_a_claim": table["seconds"].median(),
        "input_tokens_a_claim": table["input_tokens"].mean(),
        "cached_share": table["cached_tokens"].sum() / max(table["input_tokens"].sum(), 1),
        "failed_claims": table["failed"].sum(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]

## The dataset

The lab's `harder-claims` dataset in Langfuse, with the handler's decision as each claim's expected output. If it isn't there yet, this cell makes it, as the lab does.

`N_TIMES` is the number of runs of each experiment. `TINY = True` runs 5 claims for a quick check, and isn't saved.

In [ ]:
DATASET = "harder-claims"

try:
    dataset = langfuse.get_dataset(DATASET)
except Exception:
    langfuse.create_dataset(name=DATASET, description="UnderCover's harder claims with the handlers' decisions")
    for claim in claims:
        label = labels[claim["claim_id"]]
        langfuse.create_dataset_item(
            dataset_name=DATASET,
            id=claim["claim_id"],
            input=claim,
            expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"]},
            metadata={"tiny": label["tiny"]},
        )
    dataset = langfuse.get_dataset(DATASET)

TINY = False
N_TIMES = 3

items = [item for item in dataset.items if item.metadata["tiny"]] if TINY else dataset.items
print(f"Running {len(items)} of the {len(dataset.items)} claims, {N_TIMES} times for each experiment")

## The experiments

1. **Baseline:** the full wording. The agent works out each payout itself.
2. **Payout tool:** the full wording, with the payout tool.
3. **Short wording + payout tool:** the assignment, the short wording from Week 1 Day 4 with the payout tool.

Each run is an experiment on the dataset in Langfuse. A finished run is saved to `week2day1_langfuse_results.csv` with its Langfuse experiment id, and is loaded from there instead of being run again. Delete the file to start afresh.

The runs take turns: run 1 of each experiment, then run 2 of each, then run 3. The provider's speed changed during my runs: some calls took 36 seconds where most took 1 or 2, and one failed with "Our servers are currently overloaded". Running all of one experiment and then all of the next would mix the wording up with the time of day.

In [ ]:
RESULTS = Path("week2day1_langfuse_results.csv")


def saved_run(name, run):
    """The saved table for one run of an experiment, or None."""
    if TINY or not RESULTS.exists():
        return None
    saved = pd.read_csv(RESULTS)
    table = saved[(saved["experiment"] == name) & (saved["run"] == run)]
    return table if len(table) else None


def run_once(name, description, run):
    """One run of the named agent on the dataset, from the results file if it's there."""
    table = saved_run(name, run)
    if table is not None:
        print(f"{name}, run {run}: loaded from {RESULTS}")
        return table
    recorded.clear()
    result = langfuse.run_experiment(
        name=name,
        description=description,
        data=items,
        task=lambda *, item, **kwargs: decide(AGENTS[name], item.input),
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
        metadata={"run": str(run)},
    )
    print(result.format())
    table = score_table(result.item_results).assign(experiment=name, run=run, experiment_id=result.experiment_id)
    assert len(table) == len(items), f"{len(items) - len(table)} claims are missing from the run"
    if not TINY:
        table.to_csv(RESULTS, mode="a", header=not RESULTS.exists(), index=False)
    return table


DESCRIPTIONS = {
    "Baseline": "Full wording, the agent works out each payout itself",
    "Payout tool": "Full wording, with the payout tool",
    "Short wording + payout tool": "The short wording from Week 1 Day 4, with the payout tool",
}

In [ ]:
tables = {name: [] for name in DESCRIPTIONS}
for run in range(1, N_TIMES + 1):
    for name, description in DESCRIPTIONS.items():
        tables[name].append(run_once(name, description, run))

baseline, payout_tool, short = (pd.concat(tables[name], ignore_index=True) for name in DESCRIPTIONS)

## Questions 1 and 2: accuracy, cost and speed

Each figure is the average of the runs. In Langfuse, open **Datasets**, then `harder-claims`, and select runs to compare them claim by claim.

In [ ]:
EXPERIMENTS = {"Baseline": baseline, "Payout tool": payout_tool, "Short wording + payout tool": short}


def runs_of(table):
    """The scores of each run of an experiment, one row a run."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")])


summary = pd.DataFrame({name: runs_of(table).mean() for name, table in EXPERIMENTS.items()}).T
summary.round(3)

The lowest and the highest run of each experiment, on the main measures.

In [ ]:
RANGED = ["accuracy", "decision_accuracy", "cost_of_mistakes", "wrongful_declines", "llm_cost", "seconds_a_claim"]

ranges = pd.concat({name: runs_of(table)[RANGED].agg(["min", "max"]) for name, table in EXPERIMENTS.items()})
ranges.round(3)

In [ ]:
for name, table in EXPERIMENTS.items():
    show_confusion_matrix(table, f"{name}, all {table['run'].nunique()} runs")

## Is the difference real?

With 25 claims, one claim moves the accuracy by 4 points, and the agent's answers change from run to run. Both wordings decided the same 25 claims, so we compare them claim by claim: each claim's average over its runs with one wording, minus its average with the other. A bootstrap resamples the 25 claims 10,000 times to give a 95% interval for each difference. When the interval includes zero, these runs can't tell the two apart.

The same comparison of the payout tool against the baseline shows what a clear difference looks like.

In [ ]:
MEASURES = {  # what we report: (the column, and the scale from a claim's average to the figure)
    "accuracy (%)": ("right", 100),
    "cost of mistakes a run ($)": ("mistake_cost", len(items)),
    "model cost a run ($)": ("llm_cost", len(items)),
    "seconds a claim": ("seconds", 1),
    "input tokens a claim": ("input_tokens", 1),
}
RESAMPLES = np.random.default_rng(2026).integers(0, len(items), size=(10_000, len(items)))


def per_claim(table):
    """Each claim's average over the runs."""
    return table.groupby("claim_id")[[column for column, _ in MEASURES.values()]].mean()


def compare(first, second):
    """second minus first on each measure, with a paired bootstrap 95% interval."""
    a = per_claim(EXPERIMENTS[first])
    b = per_claim(EXPERIMENTS[second]).loc[a.index]
    rows = {}
    for measure, (column, scale) in MEASURES.items():
        differences = (b[column] - a[column]).to_numpy(dtype=float)
        resampled = np.nanmean(differences[RESAMPLES], axis=1) * scale
        low, high = np.percentile(resampled, [2.5, 97.5])
        rows[measure] = {
            first: np.nanmean(a[column]) * scale,
            second: np.nanmean(b[column]) * scale,
            "difference": np.nanmean(differences) * scale,
            "95% from": low,
            "95% to": high,
            "clear?": "yes" if low > 0 or high < 0 else "no",
        }
    return pd.DataFrame.from_dict(rows, orient="index")


compare("Baseline", "Payout tool").round(3)

In [ ]:
compare("Payout tool", "Short wording + payout tool").round(3)

## Claim by claim

The claims that either wording got wrong in at least one run. `full right` and `short right` count the runs that got it right, and the answers are run by run.

In [ ]:
def answers(table):
    """Each claim's answers, run by run, such as 'approve 1,416 | refer | approve 1,416'."""
    def describe(runs):
        return " | ".join(f"approve {payout:,.0f}" if agent == "approve" else agent for agent, payout in zip(runs["agent"], runs["agent_payout"]))
    return table.sort_values("run").groupby("claim_id")[["agent", "agent_payout"]].apply(describe)


by_claim = pd.DataFrame({
    "handler": payout_tool.groupby("claim_id")["handler"].first(),
    "handler payout": payout_tool.groupby("claim_id")["handler_payout"].first(),
    "full right": payout_tool.groupby("claim_id")["right"].sum(),
    "short right": short.groupby("claim_id")["right"].sum(),
    "full answers": answers(payout_tool),
    "short answers": answers(short),
})
runs = payout_tool["run"].nunique()
by_claim[(by_claim["full right"] < runs) | (by_claim["short right"] < runs)]

For the claims where the two wordings did differently, the handler's reason and the agent's rationale on each wrong answer. These show which rule a wording lost.

In [ ]:
pd.set_option("display.max_colwidth", None)
differ = by_claim.index[by_claim["full right"] != by_claim["short right"]]

for claim_id in differ:
    print(f"{claim_id}, handler: {labels[claim_id]['decision']} ${labels[claim_id]['payout']:,.2f}. {labels[claim_id]['reason']}\n")

both = pd.concat([payout_tool.assign(wording="full"), short.assign(wording="short")])
both[both["claim_id"].isin(differ) & ~both["right"]][["claim_id", "wording", "run", "agent", "agent_payout", "rationale"]].sort_values(["claim_id", "wording", "run"])

## Question 3: the cost of a wrongful decline

`WRONGFUL_DECLINE` is the cost the lab gives a decline of a claim the handler approved or referred: $1,500, the same for every claim. It stands for complaints, appeals and lost goodwill, and isn't money paid out. Nothing in the repo says what a wrongful decline really costs UnderCover, and $1,500 is the figure the course uses throughout Week 2, so it stays as the agreed figure. Instead of choosing a new one, we ask at what figure the choice of wording would change.

Arthur JZL's observation makes that exact. An experiment's cost of mistakes is its other mistakes plus its wrongful declines times `WRONGFUL_DECLINE`, a straight line, and two experiments swap places where their lines cross.

In [ ]:
def rescore(table, wrongful_decline):
    """The same answers, with each claim's mistake cost worked out again for another cost of a wrongful decline."""
    costs = [
        mistake_cost({"decision": row.agent, "payout": row.agent_payout}, {"decision": row.handler, "payout": row.handler_payout}, wrongful_decline)
        for row in table.itertuples()
    ]
    return table.assign(mistake_cost=costs)


def cost_a_run(table, wrongful_decline):
    """The cost of mistakes, averaged over the runs."""
    return rescore(table, wrongful_decline).groupby("run")["mistake_cost"].sum().mean()


def line(table):
    """The cost of the other mistakes and the number of wrongful declines, each averaged over the runs."""
    return cost_a_run(table, 0), wrongful_declines(table).groupby(table["run"]).sum().mean()


for name, table in EXPERIMENTS.items():
    other, declines = line(table)
    print(f"{name}: ${other:,.2f} of other mistakes + {declines:.2f} wrongful declines x WRONGFUL_DECLINE, a run")

(full_other, full_declines), (short_other, short_declines) = line(payout_tool), line(short)
if full_declines == short_declines:
    cheaper = "full" if full_other < short_other else "short"
    print(f"\nBoth wordings make as many wrongful declines, so WRONGFUL_DECLINE can't change the choice: the {cheaper} wording costs less")
else:
    break_even = (short_other - full_other) / (full_declines - short_declines)
    fewer, more = ("full", "short") if full_declines < short_declines else ("short", "full")
    if break_even <= 0:
        print(f"\nThe {fewer} wording, with fewer wrongful declines, costs less at any WRONGFUL_DECLINE")
    else:
        print(f"\nBreak-even at ${break_even:,.0f}: below it the {more} wording costs less, above it the {fewer} wording")

### Five figures and a claim-size scenario

Five flat figures, and one where the cost grows with the claim: $500 for the complaint and the appeal, plus half of what the handler would have paid, so a wrongful decline of a referred claim costs $500. These are assumptions to test, not UnderCover's figures.

The last column is the share of the 10,000 bootstrap resamples of the claims in which the full wording's mistakes cost less than the short wording's. Near 100% or 0% the choice is clear. Near 50%, these runs can't settle it.

In [ ]:
def claim_size(expected):
    """A wrongful decline that costs more on a bigger claim: $500, plus half the handler's payout."""
    return 500 + 0.5 * expected["payout"]


SCENARIOS = {"$0": 0, "$500": 500, "$1,500 (agreed)": 1500, "$5,000": 5000, "$10,000": 10000, "$500 + half the payout": claim_size}


def full_cheaper_share(wrongful_decline):
    """The share of bootstrap resamples of the claims in which the full wording's mistakes cost less."""
    full_costs = rescore(payout_tool, wrongful_decline).groupby("claim_id")["mistake_cost"].mean()
    short_costs = rescore(short, wrongful_decline).groupby("claim_id")["mistake_cost"].mean().loc[full_costs.index]
    differences = (short_costs - full_costs).to_numpy(dtype=float)
    return (differences[RESAMPLES].mean(axis=1) > 0).mean()


scenarios = pd.DataFrame({label: {name: cost_a_run(table, figure) for name, table in EXPERIMENTS.items()} for label, figure in SCENARIOS.items()}).T
full_costs, short_costs = scenarios["Payout tool"], scenarios["Short wording + payout tool"]
scenarios["cheaper wording"] = np.where(full_costs < short_costs, "full", np.where(full_costs > short_costs, "short", "same"))
scenarios["full cheaper in resamples"] = [f"{full_cheaper_share(figure):.0%}" for figure in SCENARIOS.values()]
scenarios.round(2)

In [ ]:
sweep = pd.DataFrame([
    {"wrongful_decline": figure, "experiment": name, "cost_of_mistakes": cost_a_run(table, figure)}
    for figure in range(0, 10_001, 250)
    for name, table in EXPERIMENTS.items()
])

fig = px.line(
    sweep,
    x="wrongful_decline",
    y="cost_of_mistakes",
    color="experiment",
    title="Cost of mistakes a run, by the cost of a wrongful decline",
    labels={"wrongful_decline": "WRONGFUL_DECLINE ($)", "cost_of_mistakes": "Cost of mistakes a run ($)", "experiment": "Experiment"},
)
fig.add_vline(x=WRONGFUL_DECLINE, line_dash="dot", annotation_text="agreed $1,500")
fig.show()

## The new costs in Langfuse

Langfuse holds each run's scores from when it ran, at $1,500. This adds each scenario's cost of mistakes to every run as another score, so Langfuse's comparison view shows them beside the original. Each score has a fixed id, so running the cell again updates the scores instead of adding more.

In [ ]:
SCORE_NAMES = {
    "$0": "cost_of_mistakes_wd_0",
    "$500": "cost_of_mistakes_wd_500",
    "$5,000": "cost_of_mistakes_wd_5000",
    "$10,000": "cost_of_mistakes_wd_10000",
    "$500 + half the payout": "cost_of_mistakes_claim_size",
}

if not TINY:
    for name, table in EXPERIMENTS.items():
        for experiment_id, run_table in table.groupby("experiment_id"):
            for label, score_name in SCORE_NAMES.items():
                langfuse.create_score(
                    score_id=f"{experiment_id}-{score_name}",
                    name=score_name,
                    value=float(rescore(run_table, SCENARIOS[label])["mistake_cost"].sum()),
                    dataset_run_id=experiment_id,
                    comment=f"Cost of mistakes with a wrongful decline at {label}",
                )
    langfuse.flush()
    print(f"Added {len(SCORE_NAMES)} scenario costs to each of the runs in Langfuse")

## The runs as Langfuse holds them

So far the comparison comes from this notebook's tables. Here are the same runs read back from Langfuse, with its v4 experiments API, as Darko Martinovic did. Langfuse Cloud switched off the old v3 API for dataset runs on 19 November 2026. Langfuse can take a few minutes to show new scores, so run this cell again if some are missing.

In [ ]:
def langfuse_runs(experiment_ids):
    """The runs Langfuse holds for these experiment ids, with their scores."""
    runs, cursor = [], None
    while True:
        page = langfuse.api.experiments.list(
            from_start_time=datetime.now(timezone.utc) - timedelta(days=30),
            dataset_id=dataset.id,
            fields="core,scores",
            limit=100,
            cursor=cursor,
        )
        runs += page.data
        cursor = page.meta.cursor
        if not cursor:
            break
    return pd.DataFrame([
        {"experiment": run.name.rsplit(" - ", 1)[0], "started": run.start_time, "items": run.item_count, **{score.name: score.value for score in run.scores or []}}
        for run in runs
        if run.id in experiment_ids
    ]).sort_values("started")


held = langfuse_runs(set(pd.concat(EXPERIMENTS.values())["experiment_id"]))
held.reindex(columns=["experiment", "started", "items", "failed_claims", "accuracy", "cost_of_mistakes", "cost_of_mistakes_wd_5000", "cost_of_mistakes_claim_size", "seconds_a_claim", "cached_share"]).round(3)

## What I found

Three runs of each experiment on the 25 harder claims, with GPT-6 Luna, on 19 November 2026. The write-up, with the tables and the claim-by-claim causes, is in [WEEK2_DAY1_LANGFUSE.md](WEEK2_DAY1_LANGFUSE.md).

1. **The short wording doesn't keep the accuracy.** It's 81.3% against 93.3%, a drop of 12 points with a 95% interval of 1 to 24. Three sentences it leaves out cause 73% of its cost of mistakes: the rafting rule (CLM-1209, declined in all three runs), the camera with its lens as one item (CLM-1216, overpaid $328 in all three) and paying for a notebook's physical cost (CLM-1207).
2. **It's slightly cheaper and probably a little faster.** It costs 8.7% less, about $0.0009 a run, because most of the input is read from the provider's cache anyway. It's about 1.7 seconds a claim faster, most of that in one run.
3. **`WRONGFUL_DECLINE` doesn't change the choice.** The full wording has fewer wrongful declines and fewer other mistakes, so it costs less at every figure from $0 to $10,000, and in the claim-size scenario. The gap in cost of mistakes isn't settled, though: the full wording is cheaper in about 70% of bootstrap resamples.
4. **No claim failed in any run.** One claim got a 502 from the provider and succeeded on the retry.

When the last cell first ran, Langfuse hadn't yet processed the last run's scores. Read again a few minutes later, it showed all nine runs with 25 claims and every scenario score.